In [1]:
import requests
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent
from dataclasses import dataclass


In [2]:
#模型参数
model = ChatOpenAI(
    model="deepseek-chat",
    api_key="sk-fc543ac8637b4711a8324eb5cefcc636",
    base_url="https://api.deepseek.com"
)

checkpointer = InMemorySaver()

In [3]:
#API参数
API_KEY = "231ac45c4fb548049c5f9d381bea89a9"
API_HOST = "jm359g6h7e.re.qweatherapi.com"

In [4]:
#城市代码检索
@tool
def search_city(location: str, adm: str = "") -> dict:
    """根据城市名称搜索城市信息，返回城市的LocationID、经纬度等。
    用户提到任何地名时，应先调用此工具获取LocationID，再用于天气查询。

    Args:
        location: 城市名称、经纬度坐标或LocationID。支持模糊搜索，如"北京"或"beij"
        adm: 上级行政区划，用于过滤重名城市。如 location="朝阳" adm="北京" 只返回北京朝阳区
    """
    url = f"https://{API_HOST}/geo/v2/city/lookup"
    params = {"location": location, "key": API_KEY, "range": "cn"}
    if adm:
        params["adm"] = adm

    response = requests.get(url, params=params)
    data = response.json()

    if data.get("code") != "200":
        return {"error": f"城市搜索失败，状态码：{data.get('code')}"}

    results = []
    for city in data.get("location", []):
        results.append({
            "name": city["name"],
            "id": city["id"],
            "lat": city["lat"],
            "lon": city["lon"],
            "adm2": city["adm2"],
            "adm1": city["adm1"],
            "country": city["country"]
        })
    return {"cities": results}

#print(search_city.invoke({"location": "wuhan"}))

In [ ]:
#天气检索
@tool
def get_forcast_weather(location: str, hours: str) -> dict:
    """获取指定地点的实时天气数据，包括温度、体感温度、天气状况、风力风向、湿度等。

    Args:
        location: 城市的LocationID（如"101010100"）或经纬度坐标（如"116.41,39.92"）。
                  LocationID可通过search_city工具获取。
        hours: 预报小时数，支持最多168小时预报，可选值：
               24h 24小时预报。
               72h 72小时预报。
               168h 168小时预报。
    """
    url = f"https://{API_HOST}/v7/weather/{hours}"
    params = {"location": location, "key": API_KEY}

    response = requests.get(url, params=params)
    data = response.json()

    if data.get("code") != "200":
        return {"error": f"天气查询失败，状态码: {data.get('code')}"}

    hourly_list = []
    for hour in data.get("hourly", []):
        hourly_list.append({
            "fxTime": hour["fxTime"],
            "temp": f"{hour['temp']}°C",
            "text": hour["text"],
            "windDir": hour["windDir"],
            "windScale": f"{hour['windScale']}级",
            "humidity": f"{hour['humidity']}%",
            "pop": f"{hour['pop']}%",
            "precip": f"{hour['precip']}mm",
        })
    return {"hourly": hourly_list}

#print(get_forcast_weather.invoke({"location": "101200101", "hours": "24h"}))

{'hourly': [{'fxTime': '2026-04-02T13:00+08:00', 'temp': '21°C', 'text': '小雨', 'windDir': '东风', 'windScale': '1-3级', 'humidity': '72%', 'pop': '55%', 'precip': '0.28mm'}, {'fxTime': '2026-04-02T14:00+08:00', 'temp': '21°C', 'text': '小雨', 'windDir': '东风', 'windScale': '1-3级', 'humidity': '73%', 'pop': '55%', 'precip': '0.21mm'}, {'fxTime': '2026-04-02T15:00+08:00', 'temp': '21°C', 'text': '小雨', 'windDir': '东南风', 'windScale': '1-3级', 'humidity': '76%', 'pop': '70%', 'precip': '0.29mm'}, {'fxTime': '2026-04-02T16:00+08:00', 'temp': '21°C', 'text': '小雨', 'windDir': '东南风', 'windScale': '1-3级', 'humidity': '81%', 'pop': '70%', 'precip': '0.7mm'}, {'fxTime': '2026-04-02T17:00+08:00', 'temp': '21°C', 'text': '小雨', 'windDir': '东风', 'windScale': '1-3级', 'humidity': '87%', 'pop': '70%', 'precip': '0.32mm'}, {'fxTime': '2026-04-02T18:00+08:00', 'temp': '18°C', 'text': '多云', 'windDir': '东风', 'windScale': '1-3级', 'humidity': '89%', 'pop': '40%', 'precip': '0.0mm'}, {'fxTime': '2026-04-02T19:00+08:00

In [ ]:
#智能体
@dataclass
class Context:
    """自定义运行时上下文模式。"""
    user_id: str

# `thread_id` 是给定对话的唯一标识符。
config = {"configurable": {"thread_id": "1"}}

agent = create_agent(
    model=model,
    tools=[get_forcast_weather, search_city],
    context_schema=Context,
    system_prompt="你是一个天气查询助手，你要使用getlocationID工具将用户要查询的地址转化为地址代码，再使用地址代码在getweather中查询，用查询结果解答用户的问题。",
    checkpointer=checkpointer
)

response = agent.invoke(
    {"messages": [{"role": "user", "content": "武汉和成都的天气怎么样？我从武汉做高铁到成都有什么穿衣建议吗"}]},
    config=config
)

print(response["messages"][-1].content)

根据查询结果，武汉当前的天气情况如下：

**武汉实时天气（2026年3月31日 12:48）：**

- **温度：** 17°C
- **体感温度：** 14°C
- **天气状况：** 阴天
- **风力风向：** 东北风2级
- **湿度：** 68%
- **降水量：** 0.0mm
- **气压：** 1013hPa
- **能见度：** 7km

总体来说，武汉现在天气比较凉爽，阴天，体感温度比实际温度稍低一些，风力不大，湿度适中。建议您外出时适当添加衣物。
